# MIP Expected Library Contract - Playground

This notebook mirrors `mip-jupyter/expected_library.md` and provides runnable examples for the notebook-side API contract.

Design boundary:
- Calls go through `platform-backend` (`/services/...`).
- `exaflow` remains internal behind backend APIs.


In [ ]:
import mip
from mip import (
    configure,
    metadata,
    algorithms,
    experiments,
    filters,
    FederatedLogisticRegression,
    FederatedLinearRegression,
    FederatedNaiveBayes,
)

configure()  # Uses environment/token-file resolution in notebook deployments.

print("mip module:", mip.__file__)
print("Facade exports:", ", ".join(mip.__all__))
print("Wrapper classes:", FederatedLogisticRegression.__name__, FederatedLinearRegression.__name__, FederatedNaiveBayes.__name__)


## 1) Metadata Discovery

Implements the expected metadata API contract:
- `metadata.list()`
- `metadata.get_pathology("dementia:0.1")`
- `metadata.describe(...)` tree view


In [ ]:
models = metadata.list()
print(f"Available data models/pathologies: {len(models)}")

for model in models[:10]:
    code = getattr(model, "code", "<unknown>")
    version = getattr(model, "version", "<unknown>")
    label = getattr(model, "label", None) or code
    print(f"- {code}:{version} ({label})")

p = metadata.get_pathology("dementia:0.1")
print("\nSelected pathology:", p.name)
print("Root variables:", len(p.variables))

def dataset_label(ds):
    if isinstance(ds, dict):
        return ds.get("label") or "<unknown>"
    if hasattr(ds, "label"):
        return getattr(ds, "label") or "<unknown>"
    return str(ds)

print("Datasets (labels):", [dataset_label(ds) for ds in p.datasets])


In [ ]:
print(metadata.describe("dementia:0.1", max_lines=120))
print("\n--- include variables ---\n")
print(metadata.describe("dementia:0.1", include_variables=True, max_lines=120))


## 2) Algorithm Discovery


In [ ]:
algos = algorithms.list()
print(f"Available algorithms: {len(algos)}")

def algo_name(algo):
    if isinstance(algo, dict):
        return algo.get("name") or algo.get("algorithm_name") or "<unknown>"
    return getattr(algo, "name", getattr(algo, "algorithm_name", str(algo)))

for algo in algos[:20]:
    print("-", algo_name(algo))


## 3) Filters Contract

Build backend-compatible rulesets using the stable filter DSL.


In [ ]:
from mip.filters import AND, OR, EQUAL, GREATER, RULESET

rule_age = ("age", GREATER, 60)
rule_dataset = ("dataset", EQUAL, "edsd")
ruleset = RULESET([rule_age, rule_dataset], AND)

print("Rule 1:", rule_age)
print("Rule 2:", rule_dataset)
print("Ruleset:", ruleset)


## 4) Experiment Lifecycle (Guarded)

Set `RUN_CREATE_EXPERIMENT=True` to execute the create/wait/results flow.


In [ ]:
RUN_CREATE_EXPERIMENT = False

if RUN_CREATE_EXPERIMENT:
    exp = experiments.create(
        name="lr-demo",
        algorithm_name="logistic_regression",
        data_model="dementia:0.1",
        datasets=["edsd"],
        x=["lefthippocampus", "righthippocampus"],
        y=["alzheimerbroadcategory"],
        filters=None,
        parameters={"positive_class": "AD"},
    )
    print("Created experiment:", exp.uuid, exp.status)
    exp.wait(timeout=180)
    print("Final status:", exp.status)
    print("Results:\n", exp.results)
else:
    print("Set RUN_CREATE_EXPERIMENT=True to execute experiments.create(...)")


## 5) Transient (Synchronous) Run (Guarded)

Set `RUN_TRANSIENT_EXPERIMENT=True` to execute `experiments.run_transient(...)`.


In [ ]:
RUN_TRANSIENT_EXPERIMENT = False

if RUN_TRANSIENT_EXPERIMENT:
    exp = experiments.run_transient(
        name="quick-run",
        algorithm_name="linear_regression",
        data_model="dementia:0.1",
        datasets=["edsd"],
        x=["age", "lefthippocampus"],
        y=["rightamygdala"],
    )
    print("Transient status:", exp.status)
    print("Transient results:\n", exp.results)
else:
    print("Set RUN_TRANSIENT_EXPERIMENT=True to execute experiments.run_transient(...)")


## 6) sklearn-Compatible Wrapper Example (Guarded)

Set `RUN_WRAPPER_DEMO=True` to run federated logistic regression, extract sklearn parameters, and materialize a local `sklearn.linear_model.LogisticRegression`.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

RUN_WRAPPER_DEMO = False

payload = {
    "name": "lr-model",
    "data_model": "dementia:0.1",
    "datasets": ["edsd"],
    "x": ["lefthippocampus", "righthippocampus"],
    "y": ["alzheimerbroadcategory"],
    "parameters": {"positive_class": "AD"},
}

if RUN_WRAPPER_DEMO:
    result = FederatedLogisticRegression(payload)
    print("experiment uuid:", result.experiment_uuid)
    print("status:", result.status)

    sk = result.get_sklearn_params()
    fitted = sk["fitted_attributes"]
    model = LogisticRegression().set_params(**sk["set_params"])
    model.classes_ = np.asarray(fitted["classes_"])
    model.coef_ = np.asarray(fitted["coef_"], dtype=float)
    model.intercept_ = np.asarray(fitted["intercept_"], dtype=float)
    model.n_features_in_ = int(fitted["n_features_in_"])
    model.n_iter_ = np.asarray(fitted.get("n_iter_", [1]), dtype=np.int32)
    if "feature_names_in_" in fitted:
        model.feature_names_in_ = np.asarray(fitted["feature_names_in_"], dtype=object)

    df = pd.DataFrame(
        {
            "lefthippocampus": [3200.0, 3000.0, 2800.0],
            "righthippocampus": [3300.0, 2950.0, 2700.0],
        }
    )
    print("predict:", model.predict(df))
    if hasattr(model, "predict_proba"):
        print("predict_proba:\n", model.predict_proba(df))
else:
    print("Set RUN_WRAPPER_DEMO=True to execute the wrapper demo.")
    print("Payload preview:", payload)


## Notes

- Keep `RUN_*` flags disabled unless you intentionally want to submit jobs.
- All experiment calls route through `platform-backend` and then to `exaflow`.
- Use this notebook as the living contract check for `mip-jupyter` notebook API behavior.
